# trainer-subclass-extend — worked example 1: Subclass trainer to add accuracy to validation output

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `trainer-subclass-extend`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The subclass-extend pattern lets you add behavior to a parent Trainer by overriding one lifecycle method and calling `super()` to preserve the base behavior. For classification tasks, you might override `validate()` to also compute accuracy without re-implementing the loss averaging that the base already handles. The key discipline is calling `super().validate()` first so the history is already populated when your extension runs.

## Worked solution

**Step 1 – Inherit from the base and add `accuracy_history`.** In `__init__`, we call `super().__init__(...)` to wire up all five base attributes, then add `self.accuracy_history = []` for our extension.

**Step 2 – Override `validate()`.** We call `super().validate()` first. After that call returns, `self.history['val_loss']` already has the new epoch's loss appended. Now we do a second pass over the val loader to compute accuracy.

**Step 3 – Second pass for accuracy.** Under `t.inference_mode()` and `model.eval()`, we run the val loader again, accumulate correct predictions, and compute `acc = correct / total`. This is a clean separation: the base computes the loss, the subclass computes accuracy.

**Step 4 – Append to `accuracy_history`.** We store the float accuracy so callers can track it alongside val loss.

In [ ]:
import torch as t
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class BaseTrainer:
    """Minimal base trainer."""
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.step = 0
        self.history = {'train_loss': [], 'val_loss': []}

    def _step(self, x, y):
        return self.loss_fn(self.model(x), y)

    def fit(self, n_epochs):
        for _ in range(n_epochs):
            self.model.train()
            for x, y in self.train_loader:
                loss = self._step(x, y)
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                self.step += 1
                self.history['train_loss'].append(loss.item())
            self.validate()

    def validate(self):
        self.model.eval()
        total, count = 0.0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                loss = self.loss_fn(self.model(x), y)
                total += loss.item() * x.shape[0]
                count += x.shape[0]
        self.history['val_loss'].append(total / count)


class AccuracyTrainer(BaseTrainer):
    """Extends BaseTrainer with per-epoch accuracy tracking."""
    def __init__(self, model, optimizer, train_loader, val_loader, loss_fn):
        super().__init__(model, optimizer, train_loader, val_loader, loss_fn)
        self.accuracy_history = []

    def validate(self):
        # 1. Base handles loss averaging and history append.
        super().validate()
        # 2. Second pass for accuracy.
        self.model.eval()
        correct, total = 0, 0
        with t.inference_mode():
            for x, y in self.val_loader:
                preds = self.model(x).argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.shape[0]
        self.accuracy_history.append(correct / total)

# Demo
t.manual_seed(4)
X = t.randn(40, 4)
Y = (X[:, 0] > 0).long()
train_dl = DataLoader(TensorDataset(X[:32], Y[:32]), batch_size=8)
val_dl = DataLoader(TensorDataset(X[32:], Y[32:]), batch_size=8)
model = nn.Linear(4, 2)
trainer = AccuracyTrainer(model, t.optim.Adam(model.parameters()),
                          train_dl, val_dl, nn.CrossEntropyLoss())
trainer.fit(3)
print('val_loss:', trainer.history['val_loss'])
print('accuracy:', trainer.accuracy_history)